# 535 — Sports Tier 1 mechanism notebook

**Conductor:** `530_sports_pipeline.ipynb` builds the base panel; this notebook extends **Tier 1** (VECTOR / SCOUT): separate **pool quality** $Q$, **congestion** $C$, **opportunity** $O$, and **draft** $Y$, then exploratory specs toward the inverted-U.

**Run order (refer to CELL names, not indices):** **EDIT BLANK** (session log) → **CELL 0** (config + `RUN_CELL*`) → **CELL 1** → **CELL 2** → **CELL 3**. Each numbered CELL is a **markdown header** immediately followed by a **code** cell whose first line is `# CELL N — …`, matching the header (same pattern as `530_sports_pipeline.ipynb`; **CELL 0** switchboard follows `540_tenure_pipeline.ipynb`).

**Theory / memos (repo):**
- `sports/documents/tier_1_roadmap.md` — **living checklist:** pipeline steps, `df` column contract, CELL 0 knobs (edit with the project).
- `1-Various_PDE_and_Chat_stuff/5-Manuscript/Vector_to_Scout_Tier1_Modeling_Direction.md`
- `1-Various_PDE_and_Chat_stuff/5-Manuscript/tier_1_model.md`
- `1-Various_PDE_and_Chat_stuff/5-Manuscript/2026_0430_Paper7_feedback.md` (theory vs minimal model; distinction scarcity)

**Data:** `datasets/mbb/player_season_panel_530.csv` via `sports_pipeline.paths.panel_530_csv()` (sports repo root).

### EDIT BLANK · session log

Burn cell for `.cursor/rules` diff sync. **Next code cell** repeats the same name in its first comment line. Not a numbered CELL.


In [ ]:
# EDIT BLANK — session log
##### EDIT BLANK ###### DISCARD
# session: 2026-05-05 — tier_1_roadmap → sports/documents; title cell path sync.
# session: 2026-03-31 — Add sports/tier_1_roadmap.md + title link.
# session: 2026-03-31 — Restore CELL 3 / CELL 0 comments (min_minutes, PRIMARY_POOL_MODE, tier1 notes).
# session: 2026-03-31 — CELL 0 md+code (config, RUN_CELL*); wrap CELL 1–3; 540-style switchboard.
# session: 2026-03-31 — 535: CELL 1–3 md/code pairs complete; roadmap refreshed; title run-order blurb.
# session: 2026-05-04 — 535: CELL headers + md/code pairs (match 530 naming).
# session: 2026-05-04 — Resync: follow .cursor/rules — blank edit first; use EditNotebook for cell content (not MCP notebook_edit_cell / bulk_add for substantive cells).
# session: 2026-03-31 — Initial 535 scaffold (rename 545→535; burn cell for diff sync).


### CELL 0 — Config · `RUN_CELL*` flags · analysis knobs

Central switchboard (same role as **`540_tenure_pipeline.ipynb`** CELL 0): which stages run, plus Tier 1 / panel options consumed by **CELL 1–3**. Re-run this code cell after changing any flag or knob.


In [ ]:
# CELL 0 — Config · RUN_CELL* flags · analysis knobs

# --- Run gates (edit here only; re-run CELL 0 after changes) ---
RUN_CELL1 = True  # Environment · imports · panel path
RUN_CELL2 = True  # Load panel · PPM · legacy poolq_loo
RUN_CELL3 = True  # Tier 1 mechanism variables

# --- Panel / Tier 1 knobs (read by CELL 2–3) ---
PERF_METRIC = "ppm"  # passed to assign_perf_from_metric in CELL 2

# min_minutes=None keeps full sample; set e.g. 50.0 later to match analysis filters.
# TODO: align with PipelineConfig.min_minutes when you want a single source of truth.
MIN_MINUTES_TIER1 = None

# Toggle for regressions: "quality" -> Q column, "crowding" -> C column (tier1_primary_pool_column).
PRIMARY_POOL_MODE = "quality"

# Weighted crowding column (TIER1_CROWDING_WEIGHTED_COL); optional / off-path for some specs.
COMPUTE_WEIGHTED_CROWDING = True

print(
    "CELL 0 ready:",
    f"RUN_CELL1={RUN_CELL1}",
    f"RUN_CELL2={RUN_CELL2}",
    f"RUN_CELL3={RUN_CELL3}",
)
print("hello world")

## CELL 1 — Environment · imports · panel path

## Phase 0 — Environment

- Run **CELL 0** first each session (`RUN_CELL*` flags and Tier 1 knobs).
- Run from **`sports/`** as cwd (or `os.chdir` in CELL 1) so `sports_pipeline.paths` resolves to `sports/datasets/mbb/`.
- Use conda env **`sports_net`** (or your gameplan env). Optional: `pip install -e .` from the **workspace root** if `import sports_pipeline` fails.

## Roadmap (VECTOR Tier 1)

1. Load panel; set `perf` (**PPM**); `recompute_teammate_loo_pool_quality` → legacy `poolq_loo` (compare to `congestion_quality`).
2. **`tier1_mechanism_vars`:** `congestion_crowding` ($C^{sum}$), `congestion_quality` ($Q$); optional weighted column; $C^{\tau}$ later.
3. Descriptives + logistic / LPM probes; partial dependence on $Q$ at fixed $C,O$.

In [ ]:
# CELL 1 — Environment · imports · panel path
from __future__ import annotations

if RUN_CELL1:
    import os
    from pathlib import Path

    # If you open Jupyter with cwd = workspace parent, uncomment to anchor sports root:
    # _SPORTS = Path("/path/to/Cursor Workspace PDE/sports").resolve()
    # os.chdir(_SPORTS)

    import pandas as pd

    from sports_pipeline import paths
    from sports_pipeline.panel_build import (
        load_panel,
        assign_perf_from_metric,
        recompute_teammate_loo_pool_quality,
    )
    from sports_pipeline.config import PipelineConfig

    CFG = PipelineConfig()
    PANEL_PATH = paths.panel_530_csv()
    print("Panel CSV:", PANEL_PATH)
    print("Exists:", PANEL_PATH.is_file())
else:
    print("  CELL 1 skipped  (RUN_CELL1 = False in CELL 0)")


### CELL 2 — Load panel · PPM · legacy `poolq_loo`

Builds `df` and baseline leave-one-out mean teammate PPM (`poolq_loo`). Tier 1 **`congestion_quality`** is recomputed in CELL 3 with a stricter valid-`perf` rule.


In [ ]:
# CELL 2 — Load panel · PPM · legacy `poolq_loo`

if RUN_CELL2:
    if "CFG" not in globals():
        print("  CELL 2 skipped — CFG undefined; run CELL 0 then CELL 1.")
    else:
        df = load_panel(CFG)
        df = assign_perf_from_metric(df, PERF_METRIC)
        df = recompute_teammate_loo_pool_quality(
            df,
            poolq_winsor_quantiles=CFG.poolq_winsor_quantiles,
        )

        assert "Y_draft" in df.columns, "Panel must include Y_draft"
        assert "poolq_loo" in df.columns
        assert "minutes" in df.columns

        df.head()
else:
    print("  CELL 2 skipped  (RUN_CELL2 = False in CELL 0)")


### CELL 3 — Tier 1 mechanism variables (`congestion_quality` · `congestion_crowding`)

Adds Tier 1 columns via `add_tier1_mechanism_variables`; `congestion_quality` uses a stricter valid-`perf` rule than legacy `poolq_loo`.


In [ ]:
# CELL 3 — Tier 1 mechanism variables (`congestion_quality` · `congestion_crowding`)
# Knobs (MIN_MINUTES_TIER1, PRIMARY_POOL_MODE, COMPUTE_WEIGHTED_CROWDING) are defined in CELL 0.

if RUN_CELL3:
    if "df" not in globals():
        print("  CELL 3 skipped — df not defined; run CELL 2 or set RUN_CELL2 = True.")
    else:
        # Q (LOO mean with valid perf), C sum, optional weighted C — see tier1_mechanism_vars.py
        from sports_pipeline.tier1_mechanism_vars import (
            add_tier1_mechanism_variables,
            TIER1_QUALITY_COL,
            TIER1_CROWDING_COL,
            TIER1_CROWDING_WEIGHTED_COL,
            tier1_primary_pool_column,
        )

        df = add_tier1_mechanism_variables(
            df,
            min_minutes=MIN_MINUTES_TIER1,
            minutes_col="minutes",
            perf_col="perf",
            compute_weighted_crowding=COMPUTE_WEIGHTED_CROWDING,
        )

        # PRIMARY_POOL_MODE in CELL 0: picks pool column for regressions (Q vs C).
        primary_x = tier1_primary_pool_column(PRIMARY_POOL_MODE)
        print(TIER1_QUALITY_COL, TIER1_CROWDING_COL, TIER1_CROWDING_WEIGHTED_COL, "primary_x=", primary_x)
        df[["perf", "poolq_loo", TIER1_QUALITY_COL, TIER1_CROWDING_COL, TIER1_CROWDING_WEIGHTED_COL]].head()
else:
    print("  CELL 3 skipped  (RUN_CELL3 = False in CELL 0)")


In [ ]:
globals()